In [1]:
from pathlib import Path
from datetime import datetime, UTC
import re

MODEL_DIR = Path(
    "/kaggle/input/models/jek1wantaufik/buddy/scikitlearn/agric/2"
)

OUTPUT = Path("/kaggle/working/submission.py")

MODULES = [
    "state.py",
    "board.py",
    "actions.py",
    "economy.py",
    "market.py",
    "scheduler.py",
    "search.py",
    "planner.py",
    "agent.py",
]

HEADER = f"""# ==========================================================
# AUTO GENERATED SUBMISSION
# Generated : {datetime.now(UTC).isoformat()}
# Source    : {MODEL_DIR}
# ==========================================================

"""

internal_modules = {Path(m).stem for m in MODULES}


def split_source(text):

    imports = []
    body = []

    blank = False

    for line in text.splitlines():

        s = line.strip()

        if s.startswith("from "):

            m = re.match(
                r"from\s+([A-Za-z0-9_]+)\s+import",
                s,
            )

            if m and m.group(1) in internal_modules:
                continue

            if line not in imports:
                imports.append(line)
            continue

        if s.startswith("import "):
            if line not in imports:
                imports.append(line)
            continue

        if s == "":
            if blank:
                continue
            blank = True
        else:
            blank = False

        body.append(line.rstrip())

    while body and body[-1] == "":
        body.pop()

    return imports, body


all_imports = []
seen_imports = set()
merged = []

print("=" * 60)
print("Building submission.py")
print("=" * 60)

for module in MODULES:

    path = MODEL_DIR / module

    if not path.exists():
        raise FileNotFoundError(path)

    print(f"✓ {module}")

    imports, body = split_source(
        path.read_text(encoding="utf-8")
    )

    for imp in imports:
        if imp not in seen_imports:
            seen_imports.add(imp)
            all_imports.append(imp)

    merged.extend(body)
    merged.append("")

while merged and merged[-1] == "":
    merged.pop()

with OUTPUT.open("w", encoding="utf-8") as f:

    f.write(HEADER)

    for imp in sorted(all_imports):
        f.write(imp + "\n")

    f.write("\n")

    for line in merged:
        f.write(line + "\n")

compile(
    OUTPUT.read_text(encoding="utf-8"),
    "submission.py",
    "exec",
)

print()
print("=" * 60)
print("SUCCESS")
print("=" * 60)
print(f"Output  : {OUTPUT}")
print(f"Modules : {len(MODULES)}")
print(f"Imports : {len(all_imports)}")
print(f"Lines   : {len(OUTPUT.read_text(encoding='utf-8').splitlines()):,}")
print(f"Size    : {OUTPUT.stat().st_size:,} bytes")

Building submission.py
✓ state.py
✓ board.py
✓ actions.py
✓ economy.py
✓ market.py
✓ scheduler.py
✓ search.py
✓ planner.py
✓ agent.py

SUCCESS
Output  : /kaggle/working/submission.py
Modules : 9
Imports : 5
Lines   : 961
Size    : 18,806 bytes


In [2]:
from kaggle_environments import make
from submission import agent

env = make("kaggriculture", debug=True)
env.run([agent, "random"])

print(env.steps[-1][0].status)
print(env.steps[-1][0].reward)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environment